## 2.2. 数据预处理
为了能用深度学习来解决现实世界的问题，我们经常从预处理原始数据开始， 而不是从那些准备好的张量格式数据开始。 在Python中常用的数据分析工具中，我们通常使用pandas软件包。 像庞大的Python生态系统中的许多其他扩展包一样，pandas可以与张量兼容。 本节我们将简要介绍使用pandas预处理原始数据，并将原始数据转换为张量格式的步骤。 后面的章节将介绍更多的数据预处理技术。


### 2.2.1. 读取数据集
举一个例子，我们首先创建一个人工数据集，并存储在CSV（逗号分隔值）文件 ../data/house_tiny.csv中。 以其他格式存储的数据也可以通过类似的方式进行处理。 下面我们将数据集按行写入CSV文件中

- 注：os 是 Python 标准库中一个非常核心和常用的模块，它提供了与操作系统（Operating System）进行交互的丰富功能。

In [1]:
import os
os.makedirs(os.path.join('..', 'data'), exist_ok=True)#创建一个名为data的目录，如果已经存在则不执行任何操作
data_file = os.path.join('..', 'data', 'house_tiny.csv')#定义一个变量data_file，表示一个名为house_tiny.csv的文件的路径
with open(data_file, 'w') as f:#以写入模式打开刚才定义的路径
    f.write('NumRooms,Alley,Price\n')  #列名
    f.write('NA,Pave,127500\n')  # 每行表示一个数据样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

要从创建的CSV文件中加载原始数据集，我们导入pandas包并调用read_csv函数。该数据集有四行三列。其中每行描述了房间数量（“NumRooms”）、巷子类型（“Alley”）和房屋价格（“Price”）。

In [7]:
import pandas as pd
data = pd.read_csv(data_file)
data

,NumRooms,Alley,Price
0,NaN,Pave,127500
1,2.0,NaN,106000
2,4.0,NaN,178100
3,NaN,NaN,140000


### 2.2.2. 处理缺失值
注意，“NaN”项代表缺失值。 为了处理缺失的数据，典型的方法包括插值法和删除法， 其中插值法用一个替代值弥补缺失值，而删除法则直接忽略缺失值。 在这里，我们将考虑插值法。

通过位置索引iloc，我们将data分成inputs和outputs， 其中前者为data的前两列，而后者为data的最后一列。 对于inputs中缺少的数值，我们用同一列的均值替换“NaN”项。

- 补充：iloc的用法：
- 
    常用用法
    1. 选择行：

    data.iloc[0] - 选择第一行
    data.iloc[1:3] - 选择第二行到第三行（不包括第四行）

    2. 选择列：

    data.iloc[:, 0] - 选择第一列
    data.iloc[:, 1:3] - 选择第二列到第三列

    3. 同时选择行和列：

    data.iloc[0, 0] - 选择第一行第一列的值
    data.iloc[1:3, 0:2] - 选择第二到第三行，第一到第二列

    4. 选择特定行和列：

    data.iloc[[0, 2], [1, 3]] - 选择第1和第3行，第2和第4列

In [19]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]
inputs = inputs.fillna(inputs.mean(numeric_only=True))
inputs

,NumRooms,Alley
0,3.0,Pave
1,2.0,NaN
2,4.0,NaN
3,3.0,NaN


In [23]:
inputs = pd.get_dummies(inputs, dummy_na=True)
# 获取最后两列的列名
last_two_columns = inputs.columns[1:3]
# 将最后两列的布尔值转换为0/1
inputs[last_two_columns] = inputs[last_two_columns].astype(int)

inputs

,NumRooms,Alley_Pave,Alley_nan
0,3.0,1,0
1,2.0,0,1
2,4.0,0,1
3,3.0,0,1


### 2.2.3. 转换为张量格式
现在inputs和outputs中的所有条目都是数值类型，它们可以转换为张量格式。 当数据采用张量格式后，可以通过在 2.1节中引入的那些张量函数来进一步操作。

In [26]:
import torch

X = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(outputs.values,dtype=torch.float64)
X, y

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

练习

In [28]:
# 删除缺失值最多的列（那就是第一列和第二列）
data.drop(columns=data.columns[[0, 2]], inplace=True)
data


,Alley
0,Pave
1,NaN
2,NaN
3,NaN
